# ANANTA — Free-first Gemma 4 training notebook

This notebook prepares the approved Khmer dataset, checks the available GPU, fine-tunes a Gemma 4 E2B LoRA adapter, and runs the same 20-case benchmark against the base model and adapter.

**Cost rule:** This notebook does not create paid cloud resources or call paid APIs. Use a free GPU runtime only if Google Colab offers one. GPU availability is not guaranteed. Stop if no GPU is available.

**Model access:** Accept the Gemma model terms on Hugging Face and provide a read token when prompted. The token stays in this notebook session; never paste it into GitHub or the source files.

**Important:** The current 150 records are teacher-authored synthetic examples from 15 approved lessons. They are a pilot dataset, not real customer conversations. The script enforces minimum split sizes, but human review is still required before deployment.

In [ ]:
# Step 1 — Check that Colab actually assigned a free CUDA GPU
import torch
assert torch.cuda.is_available(), 'No GPU was assigned. In Runtime > Change runtime type, choose a GPU if a free option is available. Do not purchase compute.'
gpu = torch.cuda.get_device_name(0)
vram = torch.cuda.get_device_properties(0).total_memory / (1024**3)
print(f'GPU: {gpu} | VRAM: {vram:.1f} GB')
assert vram >= 7, 'This GPU has less than the approximate 7 GB VRAM requirement for Gemma 4 E2B QLoRA. Stop here.'

In [ ]:
# Step 2 — Get the current project source from GitHub
import os, pathlib
if not pathlib.Path('/content/7inOne/.git').exists():
    !git clone https://github.com/klarjas/7inOne.git /content/7inOne
%cd /content/7inOne
!git pull --ff-only


In [ ]:
# Step 3 — Install the training dependencies
!pip -q install -r scripts/ananta/requirements-train.txt

In [ ]:
# Step 4 — Validate the approved data and create leakage-safe splits
!python scripts/ananta/build_dataset.py docs/ananta-sft-pilot-v0.1.jsonl --output-dir artifacts/ananta-dataset
import json
manifest = json.load(open('artifacts/ananta-dataset/manifest.json', encoding='utf-8'))
print(json.dumps(manifest, ensure_ascii=False, indent=2))
assert manifest['training_ready'], 'Dataset gate failed. Do not start training.'

In [ ]:
# Step 5 — Hugging Face access token for downloading Gemma 4 only
# Create a read-only token at https://huggingface.co/settings/tokens after accepting the Gemma model terms.
# The token is kept in this runtime memory only and is never committed to GitHub.
from getpass import getpass
import os
os.environ['HF_TOKEN'] = getpass('Paste Hugging Face READ token (input is hidden): ')
assert os.environ['HF_TOKEN'].strip(), 'A Hugging Face read token is required to download the model.'

In [ ]:
# Step 6 — Train ANANTA's LoRA adapter (may take time; GPU only)
!python scripts/ananta/train_lora.py --data-dir artifacts/ananta-dataset --output-dir artifacts/ananta-adapter --model-id google/gemma-4-E2B-it --epochs 3 --max-length 1024

In [ ]:
# Step 7 — Run the fixed benchmark against the unmodified base model
!python scripts/ananta/run_local_benchmark.py --model-id google/gemma-4-E2B-it --load-in-4bit --benchmark docs/ananta-benchmark-v0.1.jsonl --output artifacts/ananta-evaluations/base.jsonl

In [ ]:
# Step 8 — Run the exact same benchmark against ANANTA's trained adapter
!python scripts/ananta/run_local_benchmark.py --model-id google/gemma-4-E2B-it --adapter-dir artifacts/ananta-adapter --load-in-4bit --benchmark docs/ananta-benchmark-v0.1.jsonl --output artifacts/ananta-evaluations/adapter.jsonl

In [ ]:
# Step 9 — Package the adapter and raw evaluation results for download
!cd artifacts && zip -qr /content/ANANTA-adapter-and-evaluations.zip ananta-adapter ananta-evaluations
from google.colab import files
files.download('/content/ANANTA-adapter-and-evaluations.zip')

## After the run

Do not deploy the adapter automatically. First review all 20 base answers and all 20 adapter answers against the `must_include` and `must_not_claim` fields in `docs/ananta-benchmark-v0.1.jsonl`. Confirm that Khmer quality improves and that the model does not invent prices, stock, specifications, warranty, seller status, or completed actions. Keep the adapter private until it passes review.
